# SmolVLA：真实推理 Trace 与固定噪声的视觉扰动实验

整理自 2026-10-23 的 [Colab 实验笔记本](PRIVATE_EXPERIMENT_LINK_REMOVED)。
保留自己编写的字典复制、输入检查、三次预测和 MAE 实现；旧评测与视频代码放在末尾历史区。

**按目的选择单元，不要直接“全部运行”：**

| 当前目的 | 执行路线 |
| --- | --- |
| 环境已经可用，云盘已有 `.pt` | 第 2 节挂载云盘 → 第 4 节；不必重新跑 LIBERO |
| 新建 Colab 会话、尚未安装依赖 | 按需完成第 1 节 → 第 2 节 → 第 4 节 |
| 重新采集一次真实推理 | 第 1、2 节 → 第 3 节；使用新的实验目录 |

第 3 节在**子进程**加载模型；第 4 节在 **Notebook 内核**加载模型，两个 `policy` 互不共享。
只采集 Trace 时可以跳过第 4 节的模型加载，避免同时保留两份模型占用显存。
代码单元的输出与执行编号已清空；云端实测结果单独注明，避免把历史输出误当作本地重跑结果。

## 1．按需准备运行环境

已有正常环境时跳过 clone、安装与卸载操作。新建环境时先启用 Colab GPU。
这套配置复用云端的 `v0.6.1` 安装方式；版本名之外还要记录实际源码 commit。

In [ ]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU VRAM (GB):",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
    )

assert sys.version_info >= (3, 12), "LeRobot v0.6.1 要求 Python >= 3.12"
assert torch.cuda.is_available(), "请先启用 Colab GPU 运行时"

In [ ]:
# 仅在 /content/lerobot 尚不存在时执行。
%cd /content

!git clone --depth 1 --branch v0.6.1 \
    https://github.com/huggingface/lerobot.git \
    /content/lerobot

In [ ]:
# 仅在需要安装时执行；已有环境不必反复安装。
%pip install -e "/content/lerobot[smolvla,libero,evaluation]"

如果再次遇到 `cannot import name FqnToConfig from torchao.quantization`，先确认报错链。
此前未使用 TorchAO 量化的这套实验，通过执行 `%pip uninstall -y torchao` 后**重启内核**解决。
这是针对已遇到冲突的处理，不作为每次启动都执行的安装步骤。

下面的路径修复单元只在模型加载**之前**执行：它会清除已导入的 LeRobot 模块缓存。

In [ ]:
import sys
from pathlib import Path

source_root = "/content/lerobot/src"

assert (Path(source_root) / "lerobot" / "__init__.py").is_file()

if source_root in sys.path:
    sys.path.remove(source_root)
sys.path.insert(0, source_root)

for name in list(sys.modules):
    if name == "lerobot" or name.startswith("lerobot."):
        del sys.modules[name]

import lerobot
print("实际包文件：", lerobot.__file__)

In [ ]:
import sys
import inspect

sys.path.insert(0, "/content/lerobot/src")

from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

print(inspect.getfile(SmolVLAPolicy))

In [ ]:
print("Python 可执行文件：", sys.executable)
import torch
import lerobot
import libero
import mujoco
import shutil

from importlib.metadata import version

print("LeRobot:", version("lerobot"))
print("LeRobot path:", lerobot.__file__)
print("Lerobot name", lerobot.__name__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("LIBERO:", libero.__file__)
print("MuJoCo:", mujoco.__version__)
print("lerobot-eval:", shutil.which("lerobot-eval"))

In [ ]:
import subprocess

revision = subprocess.run(
    ["git", "-C", "/content/lerobot", "rev-parse", "HEAD"],
    capture_output=True, text=True, check=True,
)
print("LeRobot commit：", revision.stdout.strip())

## 2．项目代码与 Google Drive

只有运行第 3 节 Trace 或历史评测脚本才需要同步 robot-lab。
单纯用云盘里的 `.pt` 做第 4 节对比，不依赖 Trace 脚本。
下面 clone 与 pull 是两种情形，按需选用，不要重复 clone 已存在的目录。

In [ ]:
%cd /content

!git clone https://github.com/30thinkdiff/robot-lab.git

In [ ]:
%cd /content/robot-lab

!ls -lh studies/policy_generalization/
!git log -1 --oneline

In [ ]:
# 项目已存在且需要同步时执行；先保留 Colab 中自己的未提交修改。
!git fetch
!git pull
!git log -1 --oneline

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3．采集与保存一次真实推理（已有数据可跳过）

复用官方评测入口，只记录第一次动作块生成。`trace_day2_003` 是已有成功实验的名称；
**新采集时同时更换下面两个输出目录**，例如改为 `_004`，然后重新构建命令列表。
已有 `trace.json` 时保持拒绝覆盖，不需要删除旧实验。

Notebook 的 `sys.path` 不会自动传给子进程，因此下面显式设置 `PYTHONPATH`。

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

PROJECT = Path("/content/robot-lab")       # 改成 Colab 中 robot-lab 的实际路径
LEROBOT_SRC = Path("/content/lerobot/src")
TRACE_DIR = Path("/content/outputs/trace_day2_003")
EVAL_DIR = Path("/content/outputs/trace_day2_003_eval")
TRACE_SCRIPT = PROJECT / "studies/policy_generalization/trace/eval_with_trace.py"

assert TRACE_SCRIPT.is_file(), "先把本地 Trace 文件同步到 Colab"
assert (LEROBOT_SRC / "lerobot").is_dir(), "请核对已跑通的 LeRobot 源码路径"
assert not (TRACE_DIR / "trace.json").exists(), "请为新实验换一个 TRACE_DIR"

trace_env = os.environ.copy()
trace_env["PYTHONPATH"] = os.pathsep.join(
    [str(LEROBOT_SRC), str(PROJECT), trace_env.get("PYTHONPATH", "")]
)
trace_env["MUJOCO_GL"] = "egl"

subprocess.run(
    [sys.executable, str(TRACE_SCRIPT), "--check-env"],
    env=trace_env, check=True,
)

In [ ]:
# 可选：记录器未修改时，不必每次采集都重跑这些检查。
# 这些测试验证记录器，不会运行完整 SmolVLA；即使有 GPU 也只用小 CPU 张量。
subprocess.run(
    [sys.executable, "-m", "unittest", "discover",
     "-s", str(PROJECT / "studies/policy_generalization/trace"),
     "-p", "test_trace.py", "-v"],
    env=trace_env, check=True,
)

### 3.1 首次初始化 LIBERO，再启动评测

首次导入 `libero.libero` 可能询问数据目录。子进程没有交互输入时会出现 `EOFError`。
下面显式回答 `n`，接受默认路径；已有配置时不会重新询问。

In [ ]:
init_result = subprocess.run(
    [
        sys.executable,
        "-c",
        "import libero.libero; print('LIBERO 初始化完成')",
    ],
    env=trace_env,
    input="n\n",
    text=True,
    capture_output=True,
    check=False,
)

print("退出码：", init_result.returncode)
print(init_result.stdout)
print(init_result.stderr)
init_result.check_returncode()

In [ ]:
# 官方加载 Policy、处理器和 LIBERO；只观察首次动作生成，不改变模型输出。
# 若环境检查显示 CUDA=False，请先在 Colab 设置 GPU 运行时。
import subprocess
import sys

trace_command = [
    sys.executable, str(TRACE_SCRIPT),
    f"--trace-dir={TRACE_DIR}",
    "--policy.path=lerobot/smolvla_libero",
    "--policy.device=cuda",
    "--env.type=libero",
    "--env.task=libero_object",
    "--env.task_ids=[0]",
    "--env.observation_height=256",
    "--env.observation_width=256",
    "--env.control_mode=relative",
    '--env.camera_name_mapping={"agentview_image":"camera1","robot0_eye_in_hand_image":"camera2"}',
    "--eval.batch_size=1",
    "--eval.n_episodes=1",
    "--eval.use_async_envs=false",
    "--seed=1000",
    f"--output_dir={EVAL_DIR}",
]

# 收集 stdout / stderr，失败时先查看真正的异常。
completed = subprocess.run(
    trace_command,
    env=trace_env,
    capture_output=True,
    text=True,
    check=False,
)

print("退出码：", completed.returncode)
print("普通输出末尾：")
print(completed.stdout[-4000:])
print("错误输出末尾：")
print(completed.stderr[-8000:])
# stderr 中也包含正常日志和进度条；结合退出码判断是否失败。
completed.check_returncode()

In [ ]:
import json
from pathlib import Path

trace_file = TRACE_DIR / "trace.json"

if trace_file.is_file():
    report = json.loads(trace_file.read_text())
    print("是否抓到动作：", report.get("complete"))
    print("记录的错误：", report.get("error"))
else:
    print("还没有生成 trace.json")

### 3.2 将 Tensor 与源码位置对应

`.json` 保存摘要；`.pt` 保存完整 Tensor 或 Tensor 字典；PNG 是 batch 第 0 个样本的图像预览。
`policy_input.pt` 位于策略预处理之后，因此状态已经标准化、文本已经分词。
后面回放时不再执行完整预处理器。

In [ ]:
import json
import torch
from IPython.display import Image, display

report = json.loads((TRACE_DIR / "trace.json").read_text())
print("已抓到真实动作块：", report["complete"])
print("模型实际配置：", report["metadata"]["config"])
print("前三次调用的队列变化：", report["queue_steps"])

for name, summary in report["tensors"].items():
    print(name, "shape=", summary["shape"], "dtype=", summary["dtype"],
          "range=", (summary["min"], summary["max"]))

chunk = torch.load(TRACE_DIR / "action_chunk.pt", map_location="cpu", weights_only=True)
selected = torch.load(TRACE_DIR / "selected_action.pt", map_location="cpu", weights_only=True)
torch.testing.assert_close(chunk[:, 0], selected)
print("验证通过：首次单步动作 = 首次动作块的第0行")

for image_path in sorted(TRACE_DIR.glob("*_batch0.png")):
    print(image_path.name)
    display(Image(filename=str(image_path)))

In [ ]:
summary = report["tensors"]["action_chunk"]

print(summary["shape"])
print(summary["min"], summary["max"])

camera1 = report["tensors"]["policy_input.observation.images.camera1"]
camera2 = report["tensors"]["policy_input.observation.images.camera2"]

imaoutput = report["tensors"]["prepare_images.output.0.0"]
action_chunk = report["tensors"]["action_chunk"]
acoutput = report["tensors"]["select_action.output"]

# 修复：将大括号 {} 更改为中括号 []，避免字典作为 set 元素引发 unhashable 错误
for obj in [camera1, camera2, imaoutput, action_chunk, acoutput]:
    print(obj["shape"])
    print(obj["min"], obj["max"])

### 3.3 复制本次 Trace 和评测结果到云盘

这里保留 `copytree` 的复制方式，名称直接取自本次的两个输出路径。
`dirs_exist_ok=True` 会合并同名目录并覆盖同名文件；不同实验请使用不同名称。

In [ ]:
from pathlib import Path
import shutil

src = Path("/content/outputs")
dst = Path("/content/drive/MyDrive/robot-lab/experiments")

dst.mkdir(parents=True, exist_ok=True)

for run_name in [
    TRACE_DIR.name,
    EVAL_DIR.name,
]:
    run_src = src / run_name

    if run_src.exists():
        shutil.copytree(
            run_src,
            dst / run_name,
            dirs_exist_ok=True
        )

print("已复制现有实验结果到:", dst)
assert (dst / TRACE_DIR.name / "trace.json").is_file()
assert (dst / EVAL_DIR.name / "eval_info.json").is_file()

完成后用自己的话写下三点：

1. prepare_images 前后的 shape 与数值范围怎样变化？
2. sample_actions 输出和 _get_action_chunk 输出的最后一维为什么可能不同？
3. 前三次 select_action 分别生成了几次动作块？记录如何支持你的解释？

先把 Trace 结果保存到你自己的实验目录。单次 Trace 不代表统计意义上的成功率，
这里也不把局部 CPU 演算当成完整模型验证。


## 4．固定观测与初始噪声，比较正常图像和变暗图像

这一节可以直接从云盘数据开始，无需重新创建 LIBERO 环境。

| 组别 | 图像 | 状态、语言、初始噪声 |
| --- | --- | --- |
| A1 | 原始两路图像 | 固定 |
| A2 | 与 A1 相同 | 与 A1 相同，用于检查重复预测 |
| B | 两路图像逐元素乘 `0.6` | 与 A1 相同 |

本节输出仍在归一化动作空间；没有执行动作后处理，也没有把这些动作送进环境。

### 4.1 读取保存的输入，确认内容

`torch.load` 将文件还原成 Python 对象。这里 `policy_input.pt` 是字典，
`initial_noise.pt` 是 Tensor。`map_location="cpu"` 先放在 CPU 上便于检查。
`weights_only=True` 支持这里的 Tensor 和基本容器，不代表只能读取模型权重。

In [ ]:
import torch
from pathlib import Path

trace_dir = Path("/content/drive/MyDrive/robot-lab/experiments/trace_day2_003")

batch_cpu = torch.load(
    trace_dir / "policy_input.pt",
    map_location="cpu",
    weights_only=True,
)

noise_cpu = torch.load(
    trace_dir / "initial_noise.pt",
    map_location="cpu",
    weights_only=True,
)

In [ ]:
print(type(batch_cpu))
print(batch_cpu.keys())

for name, value in batch_cpu.items():
    print(name, value.shape, value.dtype, value.min(), value.max())

print("初始噪声：", noise_cpu.shape, noise_cpu.dtype)
print(batch_cpu["observation.state"])
print(batch_cpu["observation.images.camera1"])

### 4.2 保留自己的循环实现，只改两路图像

`clone()` 得到独立 Tensor；直接令 `batch_b_cpu = batch_cpu` 会让两个变量指向同一个字典。
亮度运算始终从 **A 组原图**读取，避免重复执行单元时从 0.6 倍继续变成 0.36 倍。
修改 CPU 输入后，需要重新执行搬到 GPU 和动作预测的单元。

In [ ]:
batch_b_cpu = {}

for name, value in batch_cpu.items():
  batch_b_cpu[name] = value.clone()

print(type(batch_b_cpu))
for name, value in batch_b_cpu.items():
    print(name, value.shape, value.dtype, value.min(), value.max())

In [ ]:
for image_key in [
    "observation.images.camera1",
    "observation.images.camera2",
]:
  batch_b_cpu[image_key] = (batch_cpu[image_key] * 0.6).clamp(0.0, 1.0)
print(batch_b_cpu["observation.images.camera1"])

In [ ]:
assert torch.equal(
    batch_cpu["observation.state"],
    batch_b_cpu["observation.state"],
)
assert torch.equal(
    batch_cpu["observation.language.tokens"],
    batch_b_cpu["observation.language.tokens"],
)
assert torch.equal(
    batch_cpu["observation.language.attention_mask"],
    batch_b_cpu["observation.language.attention_mask"],
)

### 4.3 在当前 Notebook 内核加载模型，再移动输入

如果本次内核已加载这个 checkpoint，可以跳过模型加载单元，执行配置查询确认即可。
若重启了内核，先完成第 1 节的源码路径与模型类导入。
`.to("cuda")` 移动设备，不要把语言 ID 和 bool mask 统一转换成浮点数。

In [ ]:
import inspect
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

checkpoint = "lerobot/smolvla_libero"

policy = SmolVLAPolicy.from_pretrained(checkpoint)
policy.to("cuda")
policy.eval()

In [ ]:
parameter = next(policy.parameters())
print(parameter.shape)
print(parameter.device)
print(parameter.dtype)
print(policy.config.chunk_size)
print(policy.config.n_action_steps)
print(policy.config.num_steps)
print(inspect.getfile(type(policy)))

In [ ]:
device = "cuda"

batch_a = {
    name: value.to(device)
    for name, value in batch_cpu.items()
}

batch_b = {
    name: value.to(device)
    for name, value in batch_b_cpu.items()
}

noise = noise_cpu.to(device)

### 4.4 三次都实际生成动作块

`noise.clone()` 复制同一份噪声的数值，不重新随机采样。
使用 `predict_action_chunk`，避免 `select_action` 从动作队列取缓存而跳过本次图像输入。

In [ ]:
a1 = policy.predict_action_chunk(
    batch_a,
    noise=noise.clone(),
)
a2 = policy.predict_action_chunk(
    batch_a,
    noise=noise.clone(),
)
b = policy.predict_action_chunk(
    batch_b,
    noise=noise.clone(),
)

### 4.5 自己实现的逐元素误差与 MAE

保留“作差 → 绝对值 → 平均 → `.item()`”的分步写法。
下面前两行中间量可单独打印，便于理解；函数将相同计算封装起来复用。
`[1,50,7]` 包含 350 个有效动作分量，未指定维度的 `.mean()` 对它们一起求平均。

In [ ]:
assert a1.shape == b.shape

difference = a1 - b
absolute_difference = difference.abs()

In [ ]:
def action_mae(a, b):
    assert a.shape == b.shape
    # 我的实现：作差 → 取绝对值 → 平均 → 转为 Python 数字
    difference = a - b
    absolute_difference = difference.abs()
    mean_absolute_difference = absolute_difference.mean()
    return mean_absolute_difference.item()

In [ ]:
print("动作块形状：", a1.shape)
print("相同输入重复预测的 MAE：", action_mae(a1, a2))
print("图像变暗后的 MAE：", action_mae(a1, b))

### 4.6 已保存的云端实测结果与解释边界

以下来自 2026-10-23 云端笔记本已执行单元的输出，**不是整理后在本地重跑的输出**：

```text
动作块形状： torch.Size([1, 50, 7])
相同输入重复预测的 MAE： 0.0
图像变暗后的 MAE： 0.10071975737810135
```

在这份固定观测、固定初始噪声和当前模型配置下，重复预测一致，而图像扰动改变了动作预测。
MAE 位于归一化动作空间，不是百分比、米或角度，也不能证明任务成功率下降。
这是一个观测、一个噪声样本的结果，不能代表整个任务或所有扰动强度。
云端本次输出来自亮度单元首次执行；整理后的亮度单元改为从原图计算，以保证重复执行不累乘。

本阶段到此结束，按动作维度计算误差保留为可选分析，不再以亮度扫描作为主线。
下一步转入 LIBERO-plus 的任务、语言、初始化与实际相机条件核对，
再开展最小闭环相机对照实验。任务单见 `studies/policy_generalization/libero_plus_protocol.md`。
当前尚未完成 LIBERO-plus 的安装、环境加载或相机评测。

## 5．历史实验区（按需单独运行）

### 5.1 云端早期闭环亮度试验

保留已经使用过的 baseline / identity / brightness 脚本入口与结果查看方法。
这些是机器人动作会影响下一帧观测的闭环评测，和第 4 节固定观测的比较不同。
早期对比没有显式配对每次动作生成的初始噪声，不能把全部输出差异都归因于亮度。
各实验目录可能已经存在；复跑请换名称。执行前确认 robot-lab 路径和依赖已准备好。

In [ ]:
!RUN_NAME=baseline_pilot_001 \
N_EPISODES=1 \
EVAL_DRIVER=native \
OUTPUT_ROOT=/content/outputs \
bash scripts/eval_libero.sh

In [ ]:
!find /content/outputs/baseline_pilot_001 \
    -maxdepth 4 -type f

In [ ]:
import json

path = "/content/outputs/baseline_pilot_001/eval_info.json"

with open(path) as f:
    results = json.load(f)

print("Overall:", results.get("overall"))
print("Per-task:", results.get("per_task"))

实验 B：不改变亮度的处理链检查（factor=1.0）。

In [ ]:
!RUN_NAME=brightness_identity_001 \
N_EPISODES=1 \
EVAL_DRIVER=brightness \
BRIGHTNESS_FACTOR=1.0 \
DEBUG_DIR=/content/outputs/debug_identity \
OUTPUT_ROOT=/content/outputs \
bash scripts/eval_libero.sh

实验 C：亮度 factor=0.6 的早期闭环试验。

In [ ]:
!RUN_NAME=brightness_060_pilot_001 \
N_EPISODES=1 \
EVAL_DRIVER=brightness \
BRIGHTNESS_FACTOR=0.6 \
DEBUG_DIR=/content/outputs/debug_brightness_060 \
OUTPUT_ROOT=/content/outputs \
bash scripts/eval_libero.sh

In [ ]:
!find /content/outputs/debug_brightness_060 \
    -type f

In [ ]:
from IPython.display import display, Image
from pathlib import Path

debug_dir = Path("/content/outputs/debug_brightness_060")

for path in sorted(debug_dir.glob("*.png")):
    print(path.name)
    display(Image(filename=str(path)))

In [ ]:
from pathlib import Path
import shutil

src = Path("/content/outputs")
dst = Path("/content/drive/MyDrive/robot-lab/experiments")

dst.mkdir(parents=True, exist_ok=True)

for run_name in [
    "baseline_pilot_001",
    "brightness_identity_001",
    "brightness_060_pilot_001",
]:
    run_src = src / run_name

    if run_src.exists():
        shutil.copytree(
            run_src,
            dst / run_name,
            dirs_exist_ok=True
        )

print("已复制现有实验结果到:", dst)

In [ ]:
from pathlib import Path
from IPython.display import Video, display

root = Path("/content/drive/MyDrive/robot-lab/experiments")

videos = list(
    (root / "baseline_pilot_001").rglob("eval_episode_0.mp4")
)

for video in videos:
    print("Video:", video)
    print("Size (MB):", round(video.stat().st_size / 1024**2, 2))
    display(Video(filename=str(video), embed=True))

### 5.2 本地原有评测、视频保存与播放代码

以下保留本地原有非空代码单元，便于回查，不作为当前主流程执行。
其中 `ae2b57f` checkout 是历史环境，不能与当前 `v0.6.1` 环境混跑；
旧单元中多个 Drive 挂载点也各有自己的前提，当前主流程统一使用 `/content/drive`。
`setpts=4.0*PTS` 表示播放时长变为 4 倍，是否恢复真实速度取决于原视频导出方式。

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
!git clone --depth 1 \
  https://github.com/huggingface/lerobot.git \
  /content/lerobot

!git -C /content/lerobot fetch --depth 1 origin ae2b57f
!git -C /content/lerobot checkout ae2b57f

In [ ]:
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

print("SmolVLA import OK")

import libero
import mujoco

print("LIBERO and MuJoCo imports OK")

In [ ]:
import sys
import importlib
from pathlib import Path

# 1. 找到真正的 LeRobot Python 包
LEROBOT_SRC = "/content/lerobot/src"

assert Path(
    LEROBOT_SRC + "/lerobot/datasets"
).is_dir(), "LeRobot 源码路径不正确"

# 2. 将正确源码路径放在模块搜索列表最前面
if LEROBOT_SRC in sys.path:
    sys.path.remove(LEROBOT_SRC)

sys.path.insert(0, LEROBOT_SRC)

# 3. 清理之前可能缓存的错误 lerobot 模块
# 当前是在修复导入问题，避免和其他模型程序同时运行。
for name in list(sys.modules):
    if name == "lerobot" or name.startswith("lerobot."):
        del sys.modules[name]

# 4. 刷新 Python 的模块查找缓存
importlib.invalidate_caches()

# 5. 重新测试导入
import lerobot
from lerobot.datasets import LeRobotDataset, LeRobotDatasetMetadata
from lerobot.policies.act import ACTPolicy
from lerobot.policies import make_pre_post_processors

print("LeRobot 路径:", lerobot.__file__)
print("LeRobotDataset:", LeRobotDataset)
print("ACTPolicy:", ACTPolicy)
print("make_pre_post_processors:", make_pre_post_processors)
print("\n所有模块导入成功！")

In [ ]:
!MUJOCO_GL=egl lerobot-eval \
  --policy.path=lerobot/smolvla_libero \
  --policy.device=cuda \
  --env.type=libero \
  --env.task=libero_object \
  --env.task_ids="[0]" \
  --env.observation_height=256 \
  --env.observation_width=256 \
  --env.control_mode=relative \
  --env.camera_name_mapping='{"agentview_image":"camera1","robot0_eye_in_hand_image":"camera2"}' \
  --eval.batch_size=1 \
  --eval.n_episodes=1 \
  --eval.use_async_envs=false \
  --output_dir=/content/outputs/smolvla_libero_test

In [ ]:
from pathlib import Path
from IPython.display import Video, display

video_path = Path(
    "/content/outputs/smolvla_libero_test/"
    "videos/libero_object_0/eval_episode_0.mp4"
)

print("视频是否存在:", video_path.is_file())

if video_path.is_file():
    print(
        "文件大小:",
        round(video_path.stat().st_size / 1024**2, 2),
        "MiB"
    )

    display(Video(str(video_path), embed=True))
else:
    print("未找到视频，请检查评估输出目录。")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import os
import shutil

assert os.path.ismount("/content/gdrive_real"), "Google Drive 未挂载"
assert video_path.is_file(), "仿真视频不存在"

destination = Path(
    "/content/gdrive_real/MyDrive/"
    "lerobot_experiments/smolvla_libero/"
    "eval_episode_0.mp4"
)

destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(video_path, destination)

assert destination.is_file()
assert destination.stat().st_size == video_path.stat().st_size

print("已复制视频:", destination)

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil

# 1. 确认仿真视频存在
video_path = Path(
    "/content/outputs/smolvla_libero_test/"
    "videos/libero_object_0/eval_episode_0.mp4"
)

assert video_path.is_file(), "仿真视频不存在"

# 2. 使用当前 Notebook 专用的 Google Drive 挂载点
mount_point = Path("/content/gdrive_eval")

if not os.path.ismount(mount_point):

    # 避免把已存在的普通文件目录误当作挂载目录
    if mount_point.exists() and any(mount_point.iterdir()):
        raise RuntimeError("挂载目录已有普通文件，请换一个目录名")

    drive.mount(str(mount_point))

assert os.path.ismount(mount_point), "Google Drive 挂载失败"

# 3. 指定云端保存位置
destination = (
    mount_point
    / "MyDrive"
    / "lerobot_experiments"
    / "smolvla_libero"
    / "eval_episode_0.mp4"
)

destination.parent.mkdir(parents=True, exist_ok=True)

# 4. 复制视频
shutil.copy2(video_path, destination)

# 5. 验证复制结果
assert destination.is_file()
assert destination.stat().st_size == video_path.stat().st_size

print("视频已复制到 Google Drive：", destination)

In [ ]:
from pathlib import Path
import subprocess
from IPython.display import Video, display

src = Path(
    "/content/outputs/smolvla_libero_test/"
    "videos/libero_object_0/eval_episode_0.mp4"
)

dst = src.with_name("eval_episode_0_normal_speed.mp4")

# 将视频时间戳拉长 4 倍，并按 20 fps 输出
subprocess.run([
    "ffmpeg", "-y",
    "-i", str(src),
    "-vf", "setpts=4.0*PTS",
    "-r", "20",
    "-an",
    str(dst),
], check=True, capture_output=True, text=True)

display(Video(str(dst), embed=True))

In [ ]:
from pathlib import Path
import subprocess
from IPython.display import Video, display

src = Path(
    "/content/outputs/smolvla_libero_5episodes/"
    "videos/libero_object_0/eval_episode_3.mp4"
)

dst = src.with_name("eval_episode_0_normal_speed.mp4")

# 将视频时间戳拉长 4 倍，并按 20 fps 输出
subprocess.run([
    "ffmpeg", "-y",
    "-i", str(src),
    "-vf", "setpts=4.0*PTS",
    "-r", "20",
    "-an",
    str(dst),
], check=True, capture_output=True, text=True)

display(Video(str(dst), embed=True))